# 🪨⛏️ **Matminer - Creating New Features**  

Matminer is a Python package created in 2018 with the objective of facilitate data-driven methods to analyzing and predicting materials properties [1]. So, in this notebook, the materials present at the dataset "Superconductivity Data" (unique_m.csv) of UCI Machine Learning Repository will be used as input for Matminer. 
***

### 📚 **Importing libraries**

In [1]:
import pandas as pd

from matminer.featurizers.base import MultipleFeaturizer
from matminer.featurizers.composition import (
    ElementProperty,
    ValenceOrbital,
    Stoichiometry,
    AtomicOrbitals,
    BandCenter,
    WenAlloys,
)
from pymatgen.core import Composition

c:\Users\julia24002\.conda\envs\glm_gam\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ **Loading and processing data**

The first step is to read the dataset using pandas and create a new column, ``composition``, from the material column, which contains the chemical formulas. This is done by applying the ``Composition`` class from the pymatgen library, which converts each chemical formula into a composition object that maps chemical elements (or species) to their corresponding atomic amounts.

In [2]:
df = pd.read_csv("../data/unique_m.csv")
df

,H,He,Li,Be,B,C,N,O,F,Ne,...,Au,Hg,Tl,Pb,Bi,Po,At,Rn,critical_temp,material
0,0.0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,29.00,Ba0.2La1.8Cu1O4
1,0.0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,26.00,Ba0.1La1.9Ag0.1Cu0.9O4
2,0.0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,19.00,Ba0.1La1.9Cu1O4
3,0.0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,22.00,Ba0.15La1.85Cu1O4
4,0.0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,23.00,Ba0.3La1.7Cu1O4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21258,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,2.44,Tm0.84Lu0.16Fe3Si5
21259,0.0,0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0,0,0,122.10,Tl1Ba2Ca3Cu4O11
21260,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,1.98,Nb0.8Pd0.2
21261,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,1.84,Nb0.69Pd0.31


In [3]:
df["composition"] = df["material"].apply(Composition)

Let's test if this really worked! For this, let's check if pymatgen can show us the weight of a material composition.

In [4]:
print(f"{df['composition'].iloc[0]}: {df['composition'].iloc[0].weight}")

Ba0.2 La1.8 Cu1 O4: 405.038846 amu


### 💡 **Featurizers**

The next stage uses the composition column created in the previous step to generate additional features with the `matminer` library. Since the objective is to train a model to predict the superconducting critical temperature, ($T_c$), directly from the chemical formula, the following composition-based featurizers were selected:

* `Stoichiometry`: Computes stoichiometric descriptors, including norms based on the elemental fractions in each composition.

* `ElementProperty`: Calculates statistical summaries of elemental properties using the compound stoichiometry. For each selected elemental property, it returns descriptors such as the minimum, maximum, range, mean and standard deviation. MAGPIE database, weighted by the compound stoichiometry.

* `ValenceOrbital`: Generates descriptors associated with the fractions of valence electrons occupying the (s), (p), (d), and (f) orbitals.

* `BandCenter`: Estimates the absolute position of the electronic band center from the electronegativities of the constituent elements.


In [5]:
featurizer = MultipleFeaturizer([
    Stoichiometry(),
    ElementProperty.from_preset("magpie"), 
    ValenceOrbital(),
    BandCenter(),
])
df_matminer = featurizer.featurize_dataframe(df, col_id="composition", ignore_errors=True)

c:\Users\julia24002\.conda\envs\glm_gam\lib\site-packages\matminer\utils\data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)
c:\Users\julia24002\.conda\envs\glm_gam\lib\site-packages\matminer\featurizers\composition\orbital.py:115: UserWarning: ValenceOrbital(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced b

In [6]:
df_matminer.to_csv("../data/df_matminer.csv")

With that, we can save the dataset with pandas function to convert to ".csv".

In [7]:
df_matminer.columns

Index(['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne',
       ...
       'MagpieData mode SpaceGroupNumber', 'avg s valence electrons',
       'avg p valence electrons', 'avg d valence electrons',
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'band center'],
      dtype='object', length=236)

### 🗃️ **References!**

[1] Ward, Logan, et al. “Matminer: An Open Source Toolkit for Materials Data Mining”. Computational Materials Science, v. 152, setembro de 2018, p. 60–69. https://doi.org/10.1016/j.commatsci.2018.05.018.
